# DRES client: nộp KIS và QA

Notebook này không nạp SigLIP/VLM. Có thể mở song song với `hybrid_retrieval.ipynb`, dùng kernel `legalqa`.

1. Chạy ô khởi tạo và ô đăng nhập. Session chỉ giữ trong RAM, không in ra output.
2. Chạy ô giao diện, bấm **Tải buổi thi**, chọn evaluation rồi **Xem task hiện tại**.
3. Đọc câu hỏi trên DRES Viewer, nhập đúng tên task (QA01/KIS01...) và chọn loại bài.
4. QA: **Nạp bản nháp QA** từ notebook retrieval, hoặc tự nhập đáp án.
   KIS: dán imageID/tên file từ kết quả retrieval và bấm **Xem ảnh KIS** nếu cần.
5. Bấm **Xem trước bài nộp**, kiểm tra evaluation/task/đáp án, rồi **Nộp lên DRES**.

**Chạy tất cả các ô không tự nộp bài. Chỉ nút Nộp lên DRES mới gửi bài.**
Mỗi đáp án sai có thể bị trừ điểm. Chỉ một thành viên team phụ trách nộp bài;
không chạy nhiều client cùng ghi file lịch sử. Bản nháp có thể thuộc task trước: luôn đối chiếu câu hỏi trên Viewer.

Nếu báo thiếu thư viện, chạy `%pip install -r requirements-client.txt` trong kernel rồi khởi động lại kernel nếu cần.


In [ ]:
import os
import sys
from getpass import getpass
from pathlib import Path

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "src/dres_client.py").is_file() and (p / "artifacts/manifest.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Hãy mở notebook trong thư mục DS314_Ruttrich.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dres_client import DEFAULT_URL, DresClient
from src.lexical_store import read_manifest
from src.notebook_ui import dres_panel

DATA_ROOT = PROJECT_ROOT / "data"
DRAFT_PATH = PROJECT_ROOT / "artifacts/qa_draft.json"
HISTORY_PATH = PROJECT_ROOT / "artifacts/dres_history.jsonl"
manifest_rows = read_manifest(PROJECT_ROOT / "artifacts/manifest.csv")

client = DresClient(
    base_url=os.environ.get("DRES_URL", DEFAULT_URL),
    history_path=HISTORY_PATH,
)
print("Client sẵn sàng. Chưa đăng nhập, chưa gọi API.")


## Đăng nhập

Dùng biến môi trường `DRES_USERNAME`, `DRES_PASSWORD` đặt **trước khi mở Jupyter/VS Code**, hoặc nhập ở lời nhắc bên dưới.
Mật khẩu nhập bằng `getpass` không hiển thị. Không ghi mật khẩu vào cell hoặc lưu Session ID trong output.
Không cần cài dotenv; notebook không tự đọc `.env`.

Khi nhận HTTP 401, chạy lại ô này và ô giao diện để lấy session mới. Không tự nộp lại bài bị lỗi mạng/timeout.


In [ ]:
if globals().get("_dres_panel") is not None:
    _dres_panel.close()

_dres_username = os.environ.get("DRES_USERNAME") or input("DRES username: ").strip()
_dres_password = os.environ.get("DRES_PASSWORD") or getpass("DRES password: ")
try:
    client.login(_dres_username, _dres_password)
    print("Đăng nhập thành công. Chạy ô giao diện bên dưới.")
finally:
    del _dres_password
    del _dres_username


## Giao diện nộp bài

QA gửi `text` duy nhất trong answer; ảnh bằng chứng từ bản nháp chỉ dùng để xem nội bộ.
KIS gửi `mediaItemName` và collection `castle5000`.

Client kiểm tra evaluation ACTIVE và task trước lúc chuẩn bị/lúc gửi. Nếu API có tên/ID task,
client chặn khi thông tin này đã đổi. Nếu API chỉ cung cấp loại task, cần tự kiểm tra tên task trên Viewer;
kiểm tra trước POST không loại bỏ hoàn toàn thời điểm chuyển task trên máy chủ.

Đổi bất kỳ trường bài nộp nào sẽ yêu cầu xem trước lại. File lịch sử dùng để chặn nộp lại cùng bài
đã nhận hoặc chưa rõ kết quả, kể cả sau khi khởi động lại kernel. Không xóa lịch sử để thử nộp lại.


In [ ]:
if globals().get("_dres_panel") is not None:
    _dres_panel.close()
_dres_panel = dres_panel(client, manifest_rows, DATA_ROOT, DRAFT_PATH)


## Đọc phản hồi

- `200 CORRECT`: đúng; dừng nộp cho task đó.
- `200 WRONG`: sai; kiểm tra lại ảnh và câu trả lời.
- `202 INDETERMINATE`: đã nhận, đang chờ trọng tài. Theo dõi Viewer/bảng điểm; không gửi lại vì đang chờ.
- `400`: chưa có task hoặc task đã kết thúc; xem nội dung phản hồi.
- `401`: đăng nhập lại, rồi xem trước bài nộp lại.
- `404`: kiểm tra evaluation/tài khoản.
- `412`: bài trùng, sai loại bài hoặc team đã có bài đúng; xem nội dung phản hồi.
- `UNKNOWN`: timeout/lỗi mạng/lỗi máy chủ khiến chưa rõ đã nhận hay chưa. Client không tự retry POST.

**Xem lịch sử cục bộ** hiển thị 20 dòng gần nhất trong `artifacts/dres_history.jsonl`.
Một lần nộp thường có một dòng `ATTEMPTED` và một dòng kết quả. `RECEIVED` nghĩa là DRES đã trả 200/202,
không đồng nghĩa đáp án đúng. Lịch sử không chứa password/session; không tự đồng bộ kết quả chấm sau đó.
Nguồn định dạng API: hướng dẫn DRES Castle5000 QA do BTC cung cấp. Chỉ thử nộp ở buổi luyện tập phù hợp.
